In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv('../data/raw/movie.csv')
ratings = pd.read_csv('../data/raw/rating.csv')

In [3]:
# Get unique users
unique_users = ratings['userId'].unique()

# Set random seed for reproducibility and pick 5% of users
np.random.seed(42)
sampled_users = np.random.choice(
    unique_users, 
    size=int(len(unique_users) * 0.05), 
    replace=False
)

# Filter ratings to keep only sampled users
small_ratings = ratings[ratings['userId'].isin(sampled_users)].copy()

# Filter out noise (inactive users and unpopular movies)
USER_MIN_RATINGS = 20
MOVIE_MIN_RATINGS = 5

# Keep movies with at least 5 ratings
movie_counts = small_ratings['movieId'].value_counts()
popular_movies = movie_counts[movie_counts >= 5].index
cleaner_ratings = small_ratings[small_ratings['movieId'].isin(popular_movies)]

# Keep users with at least 20 ratings
user_counts = cleaner_ratings['userId'].value_counts()
active_users = user_counts[user_counts >= 20].index
filtered_ratings = cleaner_ratings[cleaner_ratings['userId'].isin(active_users)]

print(f"Final shape of ratings: {filtered_ratings.shape}")

Final shape of ratings: (989523, 4)


In [4]:
# Filter movies to match our cleaned ratings dataset
valid_movie_ids = filtered_ratings['movieId'].unique()
filtered_movies = movies[movies['movieId'].isin(valid_movie_ids)].copy()

# Process genres for LightFM item features
# Split the string 'Action|Adventure|Sci-Fi' into a list: ['Action', 'Adventure', 'Sci-Fi']
filtered_movies['genres'] = filtered_movies['genres'].str.split('|')

# Handle missing genres (MovieLens uses '(no genres listed)')
filtered_movies['genres'] = filtered_movies['genres'].apply(
    lambda x: [] if x == ['(no genres listed)'] else x
)

In [5]:
filtered_ratings.head()

,userId,movieId,rating,timestamp
4191,36,145,3.5,2011-06-07 01:33:55
4192,36,163,3.0,2011-06-07 01:30:22
4193,36,196,3.0,2011-06-07 01:32:53
4194,36,485,1.5,2011-06-07 01:32:56
4195,36,555,2.0,2011-06-07 01:30:13


In [6]:
filtered_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


In [7]:
output_dir = '../data/preprocessed'

# Save the cleaned ratings
filtered_ratings.to_csv(f'{output_dir}/ratings_cleaned.csv', index=False)

# Save the cleaned movies and their features (genres)
filtered_movies.to_csv(f'{output_dir}/movies_cleaned.csv', index=False)